# Pharmasense — N-BEATS NCD Demand Forecasting
**Interpretable neural basis expansion for chronic disease drug demand**

## Why N-BEATS for NCDs (not TFT)
NCD demand (hypertension, diabetes, TB, anaemia) has a very different character from epidemic data:
- **Smooth and predictable** — no sudden outbreak spikes
- **Driven by 2 things only:** underlying trend (prevalence growing) + seasonal attendance patterns
- **No spatial component** — HTN doesn't spread from district A to B

TFT handles this but it's overengineered for this sub-problem.  
N-BEATS (Oreshkin et al., ICLR 2020) is purpose-built for this:

```
Each block produces:
  backcast  = what the block explains from history
  forecast  = what the block predicts for future

Interpretable stacks:
  Trend stack:      polynomial basis  → explicit trend component
  Seasonality stack: Fourier basis    → explicit seasonal component
```

**Output is decomposed:** `total_demand = trend_component + seasonal_component`  
You can show judges exactly how much of drug demand is chronic growth vs monsoon effect.
This is directly useful for procurement planning.

## Before running
1. **Runtime → T4 GPU**
2. **🔑 Secrets → `PHARMASENSE_DB_PASSWORD`**
3. **Runtime → Run All**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhi-dev99/agtechathon/blob/main/ml/notebooks/04_train_nbeats_ncd.ipynb)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q',
    'psycopg2-binary','pandas','numpy','torch','scikit-learn',
    'tqdm','matplotlib'], check=True)
import torch
print(f'CUDA: {torch.cuda.is_available()}')

In [ ]:
from google.colab import userdata
DB_CONFIG = dict(
    host='db.xwtvujwdybvjbaucshbt.supabase.co', port=5432,
    dbname='postgres', user='postgres',
    password=userdata.get('PHARMASENSE_DB_PASSWORD'), sslmode='require',
)

NCD_TYPES       = ['hypertension', 'type2_diabetes', 'tuberculosis', 'anaemia']
LOOKBACK_WEEKS  = 52    # 1 year — NCD demand has strong annual seasonality
HORIZON_WEEKS   = 4     # 4-week forecast

# N-BEATS interpretable config
TREND_DEGREE    = 3     # polynomial degree for trend stack (cubic)
HARM_ORDER      = 6     # number of harmonics in seasonality stack
HIDDEN_UNITS    = 256   # FC hidden size per block
N_BLOCKS        = 3     # blocks per stack
N_STACKS        = 2     # trend + seasonality
DROPOUT         = 0.1

BATCH_SIZE      = 2048  # large batch — many NCD series, regular patterns
EPOCHS          = 100
LEARNING_RATE   = 1e-3
EARLY_STOP_PAT  = 15

DRIVE_OUTPUT = '/content/drive/MyDrive/pharmasense_models/'
import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE} | NCDs: {NCD_TYPES}')

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

In [ ]:
# ── Load NCD demand data ───────────────────────────────────────────────────────
import psycopg2, pandas as pd, numpy as np

conn = psycopg2.connect(**DB_CONFIG)
ncd_df = pd.read_sql("""
    SELECT district_id, ncd_type, week_start_date, patients_est, demand_units
    FROM ncd_demand_weekly
    ORDER BY ncd_type, district_id, week_start_date;
""", conn)
conn.close()

ncd_df['week_start_date'] = pd.to_datetime(ncd_df['week_start_date'])
ncd_df['log_demand'] = np.log1p(ncd_df['demand_units'].clip(lower=0))

print(f'Records: {len(ncd_df):,}')
print(ncd_df.groupby('ncd_type')['log_demand'].describe().round(3))

In [ ]:
# ── Build sliding window samples (per district-NCD series) ────────────────────
# Each series: one district × one NCD type
# We normalize per series (z-score) so magnitude differences don't matter

from sklearn.preprocessing import StandardScaler

series_meta = []  # (district_id, ncd_type, scaler)
all_x, all_y = [], []

groups = ncd_df.groupby(['district_id', 'ncd_type'])
MIN_LEN = LOOKBACK_WEEKS + HORIZON_WEEKS + 5

for (dist_id, ncd), grp in groups:
    grp = grp.sort_values('week_start_date').reset_index(drop=True)
    if len(grp) < MIN_LEN:
        continue
    vals = grp['log_demand'].values.astype(np.float32)

    # Normalize per series
    mu, std = vals.mean(), vals.std() + 1e-8
    v_norm = (vals - mu) / std

    # Sliding window
    for t in range(len(v_norm) - LOOKBACK_WEEKS - HORIZON_WEEKS):
        x_win = v_norm[t : t + LOOKBACK_WEEKS]          # [LOOKBACK]
        y_win = v_norm[t + LOOKBACK_WEEKS : t + LOOKBACK_WEEKS + HORIZON_WEEKS]  # [HORIZON]
        all_x.append(x_win)
        all_y.append(y_win)

    series_meta.append({'district_id': dist_id, 'ncd_type': ncd, 'mu': mu, 'std': std})

X_arr = np.array(all_x, dtype=np.float32)
Y_arr = np.array(all_y, dtype=np.float32)
print(f'Samples: {len(X_arr):,} | X: {X_arr.shape} | Y: {Y_arr.shape}')

# Shuffle and split
idx = np.random.permutation(len(X_arr))
n_train = int(len(idx)*0.75); n_val = int(len(idx)*0.15)
tr, vl, te = idx[:n_train], idx[n_train:n_train+n_val], idx[n_train+n_val:]

X_tr = torch.tensor(X_arr[tr], device=DEVICE)
Y_tr = torch.tensor(Y_arr[tr], device=DEVICE)
X_vl = torch.tensor(X_arr[vl], device=DEVICE)
Y_vl = torch.tensor(Y_arr[vl], device=DEVICE)
X_te = torch.tensor(X_arr[te], device=DEVICE)
Y_te = torch.tensor(Y_arr[te], device=DEVICE)
print(f'Train: {X_tr.shape[0]:,} | Val: {X_vl.shape[0]:,} | Test: {X_te.shape[0]:,}')

In [ ]:
# ── N-BEATS: Interpretable Architecture ───────────────────────────────────────
import torch.nn as nn
import torch.nn.functional as F

class NBEATSBlock(nn.Module):
    def __init__(self, lookback, horizon, hidden, basis_type='generic', degree_or_harmonics=3):
        super().__init__()
        self.lookback  = lookback
        self.horizon   = horizon
        self.basis_type = basis_type

        self.fc = nn.Sequential(
            nn.Linear(lookback, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),  nn.ReLU(),
            nn.Linear(hidden, hidden),  nn.ReLU(),
            nn.Linear(hidden, hidden),  nn.ReLU(),
        )

        if basis_type == 'trend':
            # Polynomial basis: coefficients for degree-p polynomial
            self.p = degree_or_harmonics + 1
            self.theta_b = nn.Linear(hidden, self.p)
            self.theta_f = nn.Linear(hidden, self.p)
            # Fixed basis vectors
            t_back  = torch.linspace(-1, 0, lookback).unsqueeze(0)    # [1, L]
            t_fwd   = torch.linspace(0, 1, horizon).unsqueeze(0)      # [1, H]
            T_back  = torch.stack([t_back.squeeze()**i for i in range(self.p)], dim=0)  # [p, L]
            T_fwd   = torch.stack([t_fwd.squeeze()**i  for i in range(self.p)], dim=0)  # [p, H]
            self.register_buffer('T_back', T_back)
            self.register_buffer('T_fwd',  T_fwd)

        elif basis_type == 'seasonality':
            # Fourier basis: sin/cos harmonics
            H = degree_or_harmonics
            self.num_basis = 2 * H
            self.theta_b = nn.Linear(hidden, self.num_basis)
            self.theta_f = nn.Linear(hidden, self.num_basis)
            freqs   = torch.arange(1, H+1).float()                        # [H]
            t_back  = torch.linspace(0, 1, lookback)
            t_fwd   = torch.linspace(1, 1+horizon/lookback, horizon)
            S_back  = torch.cat([
                torch.cos(2*torch.pi*freqs.unsqueeze(1)*t_back.unsqueeze(0)),   # [H, L]
                torch.sin(2*torch.pi*freqs.unsqueeze(1)*t_back.unsqueeze(0))    # [H, L]
            ], dim=0)  # [2H, L]
            S_fwd   = torch.cat([
                torch.cos(2*torch.pi*freqs.unsqueeze(1)*t_fwd.unsqueeze(0)),
                torch.sin(2*torch.pi*freqs.unsqueeze(1)*t_fwd.unsqueeze(0))
            ], dim=0)  # [2H, H]
            self.register_buffer('S_back', S_back)
            self.register_buffer('S_fwd',  S_fwd)
        else:
            self.theta_b = nn.Linear(hidden, lookback)
            self.theta_f = nn.Linear(hidden, horizon)

    def forward(self, x):
        h = self.fc(x)  # [B, hidden]
        if self.basis_type == 'trend':
            th_b = self.theta_b(h)  # [B, p]
            th_f = self.theta_f(h)
            backcast = (th_b @ self.T_back)  # [B, L]
            forecast = (th_f @ self.T_fwd)   # [B, H]
        elif self.basis_type == 'seasonality':
            th_b = self.theta_b(h)
            th_f = self.theta_f(h)
            backcast = (th_b @ self.S_back)
            forecast = (th_f @ self.S_fwd)
        else:
            backcast = self.theta_b(h)
            forecast = self.theta_f(h)
        return backcast, forecast


class NBEATS(nn.Module):
    """
    N-BEATS Interpretable: Oreshkin et al. (ICLR 2020)
    'N-BEATS: Neural basis expansion analysis for interpretable time series forecasting'

    Stack 1: Trend — polynomial basis, extracts monotone + slowly varying component
    Stack 2: Seasonality — Fourier basis, extracts periodic component

    Each block produces (backcast, forecast).
    Backcast is subtracted from input (residual): subsequent blocks learn on residual.
    Final forecast = sum of all block forecasts (ensemble within architecture).
    """
    def __init__(self, lookback, horizon, hidden, n_blocks,
                 trend_degree=3, harm_order=6, dropout=0.1):
        super().__init__()
        self.dropout_layer = nn.Dropout(dropout)

        # Trend stack
        self.trend_blocks = nn.ModuleList([
            NBEATSBlock(lookback, horizon, hidden, 'trend', trend_degree)
            for _ in range(n_blocks)
        ])
        # Seasonality stack
        self.season_blocks = nn.ModuleList([
            NBEATSBlock(lookback, horizon, hidden, 'seasonality', harm_order)
            for _ in range(n_blocks)
        ])

    def forward(self, x, return_decomposition=False):
        # x: [B, LOOKBACK]
        residual = x
        trend_fcast    = torch.zeros(x.size(0), self.trend_blocks[0].horizon, device=x.device)
        season_fcast   = torch.zeros_like(trend_fcast)

        for blk in self.trend_blocks:
            backcast, forecast = blk(self.dropout_layer(residual))
            residual = residual - backcast
            trend_fcast = trend_fcast + forecast

        for blk in self.season_blocks:
            backcast, forecast = blk(self.dropout_layer(residual))
            residual = residual - backcast
            season_fcast = season_fcast + forecast

        total = trend_fcast + season_fcast
        if return_decomposition:
            return total, trend_fcast, season_fcast
        return total


model = NBEATS(
    lookback=LOOKBACK_WEEKS, horizon=HORIZON_WEEKS, hidden=HIDDEN_UNITS,
    n_blocks=N_BLOCKS, trend_degree=TREND_DEGREE, harm_order=HARM_ORDER,
    dropout=DROPOUT,
).to(DEVICE)

params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'N-BEATS | params: {params:,}')
print(f'Stacks: trend (poly deg {TREND_DEGREE}) + seasonality ({HARM_ORDER} harmonics)')

In [ ]:
# ── Train ──────────────────────────────────────────────────────────────────────
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
criterion = nn.MSELoss()  # MSE fine for smooth NCD series (no outlier spikes)

train_losses, val_losses = [], []
best_val, best_state, patience = float('inf'), None, 0

for epoch in tqdm(range(EPOCHS), desc='N-BEATS'):
    model.train()
    # Mini-batch
    perm = torch.randperm(X_tr.size(0), device=DEVICE)
    ep_loss = 0.0
    for i in range(0, X_tr.size(0), BATCH_SIZE):
        idx_b = perm[i:i+BATCH_SIZE]
        optimizer.zero_grad()
        pred = model(X_tr[idx_b])
        loss = criterion(pred, Y_tr[idx_b])
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        ep_loss += loss.item()
    train_losses.append(ep_loss / (X_tr.size(0) // BATCH_SIZE + 1))

    model.eval()
    with torch.no_grad():
        vl = criterion(model(X_vl), Y_vl).item()
    val_losses.append(vl)
    scheduler.step()

    if vl < best_val:
        best_val   = vl
        best_state = {k:v.clone() for k,v in model.state_dict().items()}
        patience   = 0
    else:
        patience += 1
        if patience >= EARLY_STOP_PAT:
            print(f'Early stop @ epoch {epoch+1} | best_val={best_val:.5f}')
            break
    if (epoch+1) % 20 == 0:
        print(f'E{epoch+1:3d} | train={train_losses[-1]:.5f} val={vl:.5f} best={best_val:.5f}')

plt.figure(figsize=(10,4))
plt.plot(train_losses, label='Train'); plt.plot(val_losses, label='Val')
plt.title('N-BEATS MSE Loss'); plt.legend()
plt.savefig(DRIVE_OUTPUT+'nbeats_loss.png'); plt.show()

In [ ]:
# ── Decomposition analysis — the interpretable payoff ─────────────────────────
model.load_state_dict(best_state)
model.eval()

# Test metrics
from sklearn.metrics import mean_absolute_error
with torch.no_grad():
    P_te = model(X_te).cpu().numpy()
Y_te_np = Y_te.cpu().numpy()
print(f'Test MAE: {mean_absolute_error(Y_te_np.ravel(), P_te.ravel()):.4f}')

# Decomposition plot for 4 example series (one per NCD)
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax_i, (ax, ncd) in enumerate(zip(axes.flat, NCD_TYPES)):
    # Find a sample for this NCD
    sample_x = X_te[:100].cpu()
    with torch.no_grad():
        total, trend, season = model(sample_x[:1].to(DEVICE), return_decomposition=True)
    t_f = trend[0].cpu().numpy()
    s_f = season[0].cpu().numpy()
    to_f = total[0].cpu().numpy()
    weeks = range(HORIZON_WEEKS)
    ax.plot(weeks, to_f,  label='Total forecast', lw=2)
    ax.plot(weeks, t_f,   label='Trend component', ls='--')
    ax.plot(weeks, s_f,   label='Seasonal component', ls=':')
    ax.set_title(f'{ncd}'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.suptitle('N-BEATS Interpretable Decomposition\n(Trend + Seasonality = Total NCD Drug Demand)', fontsize=12)
plt.tight_layout()
plt.savefig(DRIVE_OUTPUT+'nbeats_decomposition.png', dpi=120); plt.show()
print('Decomposition plot saved — this is the interpretable output for the demo')

In [ ]:
# ── Save ──────────────────────────────────────────────────────────────────────
from datetime import datetime
artifact = {
    'model_state_dict': best_state,
    'model_config': {
        'lookback': LOOKBACK_WEEKS, 'horizon': HORIZON_WEEKS,
        'hidden': HIDDEN_UNITS, 'n_blocks': N_BLOCKS,
        'trend_degree': TREND_DEGREE, 'harm_order': HARM_ORDER,
    },
    'ncd_types': NCD_TYPES,
    'series_meta': series_meta,  # normalization params per series
    'best_val_loss': best_val,
    'trained_at': datetime.now().isoformat(),
    'note': 'At inference: call model(x, return_decomposition=True) to get trend + seasonal components'
}
save_path = DRIVE_OUTPUT + 'nbeats_ncd_latest.pt'
torch.save(artifact, save_path)
print(f'Saved: {save_path}')

from google.colab import files
files.download(save_path)
print('Download started → ml/artifacts/nbeats_ncd_latest.pt')